# Architecture sweep on Colab GPU

Run all 12 architecture configurations at seeds 1?11 and 42. Results stay in Google Drive so reconnecting resumes completed seeds. This creates a separate GPU result set; compare architectures within this set and record the GPU environment.

1. On the project laptop, run `python scripts/prepare_colab_architecture.py`.
2. In Colab, choose a GPU runtime, open this notebook, and run the cells in order.
3. Upload `artifacts/architecture_colab_bundle_v2.zip` when prompted.
4. After completion, download `architecture_gpu_results.zip` from the Drive folder and import it locally with `python scripts/import_architecture_gpu.py <zip-path>`.


In [ ]:
import os, zipfile, hashlib, json
from pathlib import Path
from google.colab import files

os.environ["CL_THREADS"] = "2"
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "2"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

uploaded = files.upload()
assert list(uploaded) == ["architecture_colab_bundle_v2.zip"], "Upload the bundle made by prepare_colab_architecture.py"
repo = Path("/content/cl-ids-adversarial")
repo.mkdir(exist_ok=True)
with zipfile.ZipFile("/content/architecture_colab_bundle_v2.zip") as zf:
    zf.extractall(repo)
manifest = json.loads((repo / "bundle_manifest.json").read_text())
for relative, expected in manifest["files_sha256"].items():
    actual = hashlib.sha256((repo / relative).read_bytes()).hexdigest()
    assert actual == expected, relative
os.chdir(repo)
print(f"Verified {len(manifest['files_sha256'])} bundle files")


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import torch, yaml, shutil, io, contextlib, platform, uuid
from google.colab import drive
from src.run_experiment import run, save_results
from src.reporting import check_artifact

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
gpu_name = torch.cuda.get_device_name(0)
drive.mount("/content/drive")
output = Path("/content/drive/MyDrive/cl_ids_architecture_results")
output.mkdir(parents=True, exist_ok=True)
print("GPU:", gpu_name, "Torch:", torch.__version__, "Results:", output)


In [ ]:
seeds = [1,2,3,4,5,6,7,8,9,10,11,42]
configs = sorted(Path("configs/arch").glob("ar_*.yaml"))
assert len(configs) == 12

for config_path in configs:
    base = yaml.safe_load(config_path.read_text())
    name = base["name"]
    for seed in seeds:
        cfg = dict(base, seed=seed, device="cuda")
        summary_path = output / f"{name}_summary_seed{seed}.json"
        matrix_path = output / f"{name}_R_seed{seed}.csv"
        manifest_path = output / f"{name}_manifest_seed{seed}.json"
        if check_artifact(output, name, seed, cfg)["complete"]:
            print("SKIP", name, seed)
            continue
        for old_file in (summary_path, matrix_path, manifest_path):
            if old_file.exists():
                incomplete = output / "incomplete"
                incomplete.mkdir(exist_ok=True)
                old_file.rename(incomplete / (old_file.name + ".partial." + uuid.uuid4().hex))
        result = run(cfg)
        result["summary"].update(runtime_device="cuda", gpu_name=gpu_name, torch_version=torch.__version__)
        with contextlib.redirect_stdout(io.StringIO()):
            save_results(result, cfg, output)
        assert check_artifact(output, name, seed, cfg)["complete"]
        print("DONE", name, seed, "acc=%.4f" % result["summary"]["acc"], "wall=%.0fs" % result["summary"]["wall_sec"], flush=True)


In [ ]:
expected = {(p.stem, seed) for p in configs for seed in seeds}
for name, seed in expected:
    summary = output / f"{name}_summary_seed{seed}.json"
    matrix = output / f"{name}_R_seed{seed}.csv"
    completion = output / f"{name}_manifest_seed{seed}.json"
    cfg = dict(yaml.safe_load(next(p for p in configs if p.stem == name).read_text()), seed=seed, device="cuda")
    assert summary.exists() and matrix.exists() and completion.exists(), (name, seed)
    assert check_artifact(output, name, seed, cfg)["complete"], (name, seed)
    assert json.loads(summary.read_text())["runtime_device"] == "cuda"
run_manifest = {
    "gpu_name": gpu_name, "torch_version": torch.__version__,
    "source_commit": manifest["source_commit"],
    "worktree_dirty": manifest["worktree_dirty"],
    "cuda_version": torch.version.cuda, "python": platform.python_version(),
    "cl_threads": os.environ["CL_THREADS"],
    "bundle_sha256": hashlib.sha256(Path("/content/architecture_colab_bundle_v2.zip").read_bytes()).hexdigest(),
    "task_file_sha256": manifest["files_sha256"]["data/processed/tasks_chrono.npz"],
    "configs": [p.stem for p in configs], "seeds": seeds,
    "config_sha256": {p.stem: manifest["files_sha256"]["configs/arch/" + p.name] for p in configs},
}
(output / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))
archive_tmp = output / "architecture_gpu_results.zip.tmp"
archive = output / "architecture_gpu_results.zip"
with zipfile.ZipFile(archive_tmp, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name, seed in sorted(expected):
        for suffix in (f"_R_seed{seed}.csv", f"_summary_seed{seed}.json", f"_manifest_seed{seed}.json"):
            file = output / (name + suffix)
            zf.write(file, "results/" + file.name)
    zf.write(output / "run_manifest.json", "run_manifest.json")
os.replace(archive_tmp, archive)
print("Complete:", len(expected), "GPU runs; download", archive)
